# Open Exoplanet Discovery Lab

**Explorer question:** can we find a repeating shadow in real TESS measurements?

**Research question:** after transparent detrending, automated vetting and injection–recovery, which transit-like signals justify human inspection and independent follow-up?

> A signal produced here is an **unvalidated candidate**, not a confirmed planet. Do not announce a discovery from this notebook alone.

## 1. Install the reproducible toolkit

The cell clones a version-controlled repository so the analysis can be cited and repeated.

In [1]:
!git clone -q https://github.com/Biswajit1999/open-exoplanet-discovery-lab.git
%cd open-exoplanet-discovery-lab
!pip -q install -e .[tess]

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import lightkurve as lk

from exolab import (
    ExoplanetArchiveClient, clean_lightcurve, detrend_lightcurve,
    search_bls, vet_signal, run_injection_recovery
)
from exolab.report import write_candidate_report

/content/open-exoplanet-discovery-lab
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.2/41.2 kB 2.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.1/11.1 MB 90.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 261.1/261.1 kB 16.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.5/47.5 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 49.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.1/60.1 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 243.5/243.5 kB 14.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

/usr/local/lib/python3.12/dist-packages/lightkurve/prf/__init__.py:7: UserWarning: Warning: the tpfmodel submodule is not available without oktopus installed, which requires a current version of autograd. See #1452 for details.
  warnings.warn(


ModuleNotFoundError: No module named 'exolab'

## 2. Ask the live archive what exists

`PC` means planet candidate; it does not mean confirmed planet. The counts change as the community updates dispositions.

In [ ]:
archive = ExoplanetArchiveClient()
snapshot = archive.census()
print(json.dumps(snapshot.__dict__, indent=2))

queue = archive.candidate_queue(limit=20, max_tmag=11.5, max_radius=4.0)
display(queue[["tid", "toidisplay", "st_tmag", "pl_orbper", "pl_rade", "pl_trandep", "selection_score"]])

## 3. Choose a target and download public TESS light curves

Start with a row from the table. The notebook prefers mission-produced SPOC/TESS-SPOC light curves. Record the product list shown below in any candidate report.

In [ ]:
short_period_queue = queue[queue["pl_orbper"] <= 10].reset_index(drop=True)
selected = short_period_queue.iloc[0] if len(short_period_queue) else queue.iloc[0]
TIC_ID = int(selected["tid"])  # replace with another TIC if desired
TARGET = f"TIC {TIC_ID}"
search_result = lk.search_lightcurve(TARGET, mission="TESS")
display(search_result.table[["mission", "author", "exptime", "distance"]])

MAX_PRODUCTS = 2  # raise this after the first successful run
preferred = search_result[search_result.table["author"] == "SPOC"]
if len(preferred) == 0:
    preferred = search_result[search_result.table["author"] == "TESS-SPOC"]
if len(preferred) == 0:
    raise RuntimeError("No SPOC/TESS-SPOC light curve found. Choose another TIC or add an explicit FFI extraction step.")
collection = preferred[:MAX_PRODUCTS].download_all()
lc = collection.stitch(corrector_func=lambda item: item.normalize()).remove_nans()
time, flux, flux_err = clean_lightcurve(lc.time.value, lc.flux.value, lc.flux_err.value)
flat_flux, trend = detrend_lightcurve(time, flux, window_days=1.0)

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(time, flux, '.', ms=1.5, alpha=.35, label='normalized PDCSAP')
ax.plot(time, trend, lw=1.5, label='running-median trend')
ax.set(xlabel='Time [BTJD]', ylabel='Normalized flux', title=TARGET)
ax.legend();

## 4. Search for a repeating transit-shaped signal

BLS compares periodic box-shaped dips. Restricting period and duration is both faster and more scientifically honest than silently searching an enormous parameter space.

In [ ]:
finite = np.isfinite(flat_flux)
signal, periodogram = search_bls(
    time[finite], flat_flux[finite], flux_err[finite],
    minimum_period=0.5,
    maximum_period=min(30.0, np.ptp(time[finite]) / 2),
    durations_hours=(0.75, 1.5, 3.0, 5.0),
)
print(json.dumps(signal.as_dict(), indent=2))

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(periodogram.period, periodogram.power, color='#5b4bdb')
ax.axvline(signal.period_days, color='#e76f51', ls='--')
ax.set(xlabel='Trial period [days]', ylabel='BLS depth S/N', title='Transit search');

## 5. Try to disprove the candidate

A good vetting workflow is deliberately sceptical. Alternating eclipse depths can reveal an eclipsing binary; a strong event at phase 0.5 can reveal a secondary eclipse.

In [ ]:
vetting = vet_signal(time[finite], flat_flux[finite], signal, flux_err[finite])
print(json.dumps(vetting.as_dict(), indent=2))
print('PASSING THESE TESTS DOES NOT VALIDATE A PLANET.')

## 6. Measure what the search would miss

We inject artificial transits into the same noise and ask whether the pipeline recovers their periods. Increase the grid only after the small demonstration works.

In [ ]:
recoveries = run_injection_recovery(
    time[finite], flat_flux[finite],
    periods_days=[1.5, 3.0, 6.0, 12.0],
    depths_fraction=[0.0005, 0.001, 0.002],
    duration_hours=2.0, flux_err=flux_err[finite], seed=42,
)
recovery_table = pd.DataFrame([item.as_dict() for item in recoveries])
display(recovery_table.pivot(index='depth_fraction', columns='period_days', values='recovered'))

## 7. Export a candidate scorecard

The output deliberately says *unvalidated transit-like signal*. Before any discovery claim, inspect target pixels and difference images, cross-match Gaia neighbours and ExoFOP, compare independent reductions, and seek follow-up.

In [ ]:
figure_path, json_path = write_candidate_report(
    'outputs', TARGET, time[finite], flat_flux[finite], signal, vetting
)
print(figure_path, json_path)
display(plt.imread(figure_path))

## 8. Required review checklist

- [ ] Check the period and half/double-period aliases.
- [ ] Inspect SAP, PDCSAP and at least one independent FFI extraction.
- [ ] Inspect quality flags, momentum dumps and sector release notes.
- [ ] Examine target-pixel difference images and centroid motion.
- [ ] Query nearby Gaia DR3 sources and estimate dilution.
- [ ] Cross-match TOIs, TCEs, ExoFOP, SIMBAD and the literature.
- [ ] Repeat the signal in independent sectors or observations.
- [ ] Publish the injection–recovery map and all rejection reasons.
- [ ] Ask an experienced exoplanet researcher to review any high-priority candidate.